 Задание

Ваша задача — реализовать класс Account, который моделирует поведение банковского счёта. Этот класс должен не только выполнять базовые операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

Этап 1. Реализация базового класса Account
Класс должен быть инициализирован с параметрами:

account_holder (str): имя владельца счёта;

balance (float, по умолчанию 0): начальный баланс счёта, не может быть отрицательным.

Атрибуты:

holder: хранит имя владельца;

_balance: приватный атрибут для хранения текущего баланса;

operations_history: список или другая структура для хранения истории операций.

Важно: каждая операция должна храниться не просто как число, а как структурированная информация, например, словарь или кортеж. Минимальный набор данных для операции: тип операции ('deposit' или 'withdraw'), сумма, дата и время операции, текущий баланс после операции, статус ('success' или 'fail').

 

Этап 2. Реалирзация методов

__init__(self, account_holder, balance=0): конструктор;

deposit(self, amount): метод для пополнения счёта:

принимает сумму (должна быть положительной);

в случае успеха обновляет баланс и добавляет запись в историю операций.

withdraw(self, amount): метод для снятия средств:

принимает сумму (должна быть положительной);

проверяет, достаточно ли средств на счёте, если нет — операция не проходит, но ее попытка с статусом 'fail' все равно фиксируется в истории;

в случае успеха обновляет баланс и добавляет запись.

get_balance(self): метод, который возвращает текущий баланс.

get_history(self): метод, который возвращает историю операций.

 

Важно: продумайте, в каком формате его вернуть. Для работы с датой и временем используйте модуль datetime. Получить текущее время можно с помощью datetime.now().

 

Этап 3. Реализация наследования.

Создайте класс CreditAccount(Account), который наследует всю функциональность класса Account и добавляет новую.

Особенности кредитного счёта:

При инициализации принимает дополнительный параметр credit_limit (кредитный лимит). Баланс такого счета может быть отрицательным, но не ниже значения -credit_limit.

По запросу показывает, сколько кредитных средств еще доступно (текущий баланс + кредитный лимит).

В историю операций добавляется информацию о том, были ли использованы кредитные средства в данной операции.


In [6]:
from datetime import datetime


class Account:
    """Базовый класс банковского счёта."""

    def __init__(self, account_holder: str, balance: float = 0):
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        self.holder = account_holder
        self._balance = balance
        self.operations_history: list[dict] = []

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной")
        self._balance += amount
        self.operations_history.append({
            'type': 'deposit',
            'amount': amount,
            'date': datetime.now(),
            'balance_after': self._balance,
            'status': 'success',
        })

    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной")
        if amount > self._balance:
            self.operations_history.append({
                'type': 'withdraw',
                'amount': amount,
                'date': datetime.now(),
                'balance_after': self._balance,
                'status': 'fail',
            })
            return False
        self._balance -= amount
        self.operations_history.append({
            'type': 'withdraw',
            'amount': amount,
            'date': datetime.now(),
            'balance_after': self._balance,
            'status': 'success',
        })
        return True

    def get_balance(self) -> float:
        return self._balance

    def get_history(self) -> list[dict]:
        return self.operations_history

    def print_history(self) -> None:
        """Красивый вывод истории операций."""
        w_date, w_type, w_sum, w_status, w_bal = 20, 12, 10, 30, 10
        sep = "  "
        width = w_date + w_type + w_sum + w_status + w_bal + len(sep) * 4

        print("\n" + "=" * width)
        print(f"{'ВЫПИСКА ПО СЧЁТУ':^{width}}")
        print(f"{'Владелец: ' + self.holder:^{width}}")
        print(f"{'Текущий баланс: ' + str(self._balance) + ' ₽':^{width}}")
        print("=" * width)

        if not self.operations_history:
            print("Операции отсутствуют")
            print("=" * width + "\n")
            return

        print(f"{'Дата':<{w_date}}{sep}"
              f"{'Тип':<{w_type}}{sep}"
              f"{'Сумма':>{w_sum}}{sep}"
              f"{'Статус':<{w_status}}{sep}"
              f"{'Баланс':>{w_bal}}")
        print("-" * width)

        for record in self.operations_history:
            date_str = record['date'].strftime('%Y-%m-%d %H:%M:%S')
            operation = "Пополнение" if record['type'] == 'deposit' else "Снятие"
            is_success = record['status'] == 'success'
            status_icon = "✓" if is_success else "✗"
            status_text = f"{status_icon} {record['status']}"

            print(f"{date_str:<{w_date}}{sep}"
                  f"{operation:<{w_type}}{sep}"
                  f"{record['amount']:>{w_sum}.2f}{sep}"
                  f"{status_text:<{w_status}}{sep}"
                  f"{record['balance_after']:>{w_bal}.2f}")

        print("-" * width)
        print(f"Всего операций: {len(self.operations_history)}")
        print("=" * width + "\n")


class CreditAccount(Account):
    """Кредитный счёт: баланс может уходить в минус до -credit_limit."""

    def __init__(self, account_holder: str, balance: float = 0,
                 credit_limit: float = 0):
        super().__init__(account_holder, balance)
        self.credit_limit = credit_limit

    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной")
        # Проверяем лимит: баланс + кредитный лимит
        if amount > self._balance + self.credit_limit:
            self.operations_history.append({
                'type': 'withdraw',
                'amount': amount,
                'date': datetime.now(),
                'balance_after': self._balance,
                'status': 'fail',
                'credit_used': False,
            })
            return False

        self._balance -= amount
        # Кредит использован, если баланс стал отрицательным
        credit_used = self._balance < 0

        self.operations_history.append({
            'type': 'withdraw',
            'amount': amount,
            'date': datetime.now(),
            'balance_after': self._balance,
            'status': 'success',
            'credit_used': credit_used,
        })
        return True

    def get_available_credit(self) -> float:
        """Сколько кредитных средств ещё доступно."""
        return self._balance + self.credit_limit

    def print_history(self) -> None:
        """Вывод истории с дополнительной колонкой 'Кредит использован'."""
        w_date, w_type, w_sum, w_status, w_bal, w_credit = 20, 12, 10, 20, 10, 12
        sep = "  "
        width = (w_date + w_type + w_sum + w_status + w_bal + w_credit
                 + len(sep) * 5)

        print("\n" + "=" * width)
        print(f"{'ВЫПИСКА ПО КРЕДИТНОМУ СЧЁТУ':^{width}}")
        print(f"{'Владелец: ' + self.holder:^{width}}")
        print(f"{'Кредитный лимит: ' + str(self.credit_limit) + ' ₽':^{width}}")
        print(f"{'Текущий баланс: ' + str(self._balance) + ' ₽':^{width}}")
        print(f"{'Доступно кредита: ' + str(self.get_available_credit()) + ' ₽':^{width}}")
        print("=" * width)

        if not self.operations_history:
            print("Операции отсутствуют")
            print("=" * width + "\n")
            return

        print(f"{'Дата':<{w_date}}{sep}"
              f"{'Тип':<{w_type}}{sep}"
              f"{'Сумма':>{w_sum}}{sep}"
              f"{'Статус':<{w_status}}{sep}"
              f"{'Баланс':>{w_bal}}{sep}"
              f"{'Кредит':>{w_credit}}")
        print("-" * width)

        for record in self.operations_history:
            date_str = record['date'].strftime('%Y-%m-%d %H:%M:%S')
            operation = "Пополнение" if record['type'] == 'deposit' else "Снятие"
            is_success = record['status'] == 'success'
            status_icon = "✓" if is_success else "✗"
            status_text = f"{status_icon} {record['status']}"
            credit_info = "Да" if record.get('credit_used') else "Нет"

            print(f"{date_str:<{w_date}}{sep}"
                  f"{operation:<{w_type}}{sep}"
                  f"{record['amount']:>{w_sum}.2f}{sep}"
                  f"{status_text:<{w_status}}{sep}"
                  f"{record['balance_after']:>{w_bal}.2f}{sep}"
                  f"{credit_info:>{w_credit}}")

        print("-" * width)
        print(f"Всего операций: {len(self.operations_history)}")
        print("=" * width + "\n")


# ==============================
# Тестирование
# ==============================
if __name__ == "__main__":
    print("=" * 50)
    print("ТЕСТ 1: Обычный счёт")
    print("=" * 50)
    user = Account("Пупкин", balance=1000)
    user.deposit(500)
    user.withdraw(200)
    user.withdraw(2000)  # fail — недостаточно средств
    print(f"Баланс: {user.get_balance()} ₽")
    user.print_history()
    

    print("=" * 50)
    print("ТЕСТ 2: Кредитный счёт")
    print("=" * 50)
    credit_user = CreditAccount("Залупкин", balance=1000, credit_limit=5000)
    credit_user.deposit(500)
    credit_user.withdraw(2000)  # success, без кредита
    credit_user.withdraw(3000)  # success, с использованием кредита
    credit_user.withdraw(2000)  # fail — превышает лимит
    print(f"Баланс: {credit_user.get_balance()} ₽")
    print(f"Доступно кредита: {credit_user.get_available_credit()} ₽")
    credit_user.print_history()

ТЕСТ 1: Обычный счёт
Баланс: 1300 ₽

                                     ВЫПИСКА ПО СЧЁТУ                                     
                                     Владелец: Пупкин                                     
                                  Текущий баланс: 1300 ₽                                  
Дата                  Тип                Сумма  Статус                              Баланс
------------------------------------------------------------------------------------------
2026-11-07 14:16:11   Пополнение        500.00  ✓ success                          1500.00
2026-11-07 14:16:11   Снятие            200.00  ✓ success                          1300.00
2026-11-07 14:16:11   Снятие           2000.00  ✗ fail                             1300.00
------------------------------------------------------------------------------------------
Всего операций: 3

ТЕСТ 2: Кредитный счёт
Баланс: -3500 ₽
Доступно кредита: 1500 ₽

                                 ВЫПИСКА ПО КРЕДИТНОМУ СЧЁТУ